# Algorithm 1 — LLM-Assisted Hyperlink Reranking
## Qwen3-8B on Google Colab GPU (no Ollama, no mock)

**Paper:** *LLM-Assisted Hyperlink Recommendation for Low-Resource Wikipedia Editions*

**Pipeline:**  NGCF top-100 → batch LLM scoring (Qwen3) → normalise → fuse → top-K

> **Colab setup:** Runtime → Change runtime type → **T4 GPU** (free tier)  
> RAM required: ~18 GB GPU VRAM for fp16 · use 4-bit quant for T4 (15 GB)

---

## 1. Check GPU

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                         "--format=csv,noheader"], capture_output=True, text=True)
if result.returncode == 0:
    print("GPU:", result.stdout.strip())
else:
    print("No GPU detected. Go to: Runtime → Change runtime type → GPU (T4)")

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1024**3
    print(f"Device: {props.name}  |  VRAM: {vram:.1f} GB")
    if vram < 14:
        print("⚠️  < 14 GB VRAM → will use 4-bit quantisation (BitsAndBytes)")
    else:
        print("✅ Enough VRAM for 8B model")

## 2. Install dependencies

In [ ]:
# Core dependencies
%pip install -q transformers>=4.51 accelerate>=0.30 bitsandbytes>=0.43 tqdm requests

# Verify
import transformers, accelerate, bitsandbytes as bnb
print(f"transformers : {transformers.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"bitsandbytes : {bnb.__version__}")

## 3. Upload `app_recommendations.json`

In [ ]:
from google.colab import files as colab_files
import json, os

DATA_FILE = "app_recommendations.json"

if not os.path.exists(DATA_FILE):
    print("Upload app_recommendations.json ↓")
    uploaded = colab_files.upload()
    if DATA_FILE not in uploaded:
        # rename if uploaded with different name
        key = list(uploaded.keys())[0]
        os.rename(key, DATA_FILE)
    print(f"✅ Saved as {DATA_FILE}")
else:
    print(f"✅ {DATA_FILE} already present")

with open(DATA_FILE) as f:
    _d = json.load(f)
langs    = list(_d.keys())
n_arts   = len(_d[langs[0]])
print(f"   Languages : {langs}  |  Articles : {n_arts}")

## 4. Configuration

In [ ]:
from pathlib import Path

# Paths
OUTPUT_DIR = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR  = Path("cache");   CACHE_DIR.mkdir(exist_ok=True)

# Languages
LANGUAGES   = ["en", "vi", "ja"]
LABEL_INDEX = {"en": 0, "vi": 1, "ja": 2}
LANG_NAMES  = {"en": "English", "vi": "Vietnamese", "ja": "Japanese"}

# Algorithm 1 hyperparameters (paper)
TOP_N      = 100          # NGCF candidates per article
BATCH_SIZE = 20           # candidates per LLM call
FINAL_K    = 10           # top-K to return
ALPHA      = 0.5          # fusion weight α
ALPHA_GRID = [0.3, 0.5, 0.7]
HELD_RANK  = 10           # GT = NGCF rank-10
K_VALUES   = [1, 5, 10]

# Qwen3 model
# Options (choose based on VRAM):
#   "Qwen/Qwen3-8B"    → ~16 GB fp16  |  ~8 GB 4-bit  (T4 OK with 4-bit)
#   "Qwen/Qwen3-14B"   → ~28 GB fp16  |  ~14 GB 4-bit (A100 needed for fp16)
#   "Qwen/Qwen3-0.6B"  → ~1.2 GB      (fast, less capable)
MODEL_NAME   = "Qwen/Qwen3-8B"
USE_4BIT     = True       # ← True for T4 (15 GB VRAM), False for A100/H100
USE_CACHE    = True       # disk cache LLM responses

# Thinking mode: MUST be False for clean JSON output
# (Qwen3 with thinking=True wraps output in <think>...</think> blocks)
THINKING_MODE = False

# Paper system prompt (verbatim §3.5)
SYSTEM_PROMPT = (
    "You are an experienced Wikipedia editor. "
    "Evaluate whether each candidate hyperlink should be inserted into the target article. "
    "Consider semantic relevance, topical consistency, and Wikipedia linking conventions. "
    "Ignore popularity unless supported by the article context."
)

print("✅ Configuration loaded")
print(f"   Model      : {MODEL_NAME}  ({'4-bit quant' if USE_4BIT else 'fp16'})")
print(f"   TOP_N={TOP_N}  BATCH={BATCH_SIZE}  TOP_K={FINAL_K}  α={ALPHA_GRID}")
print(f"   Thinking   : {THINKING_MODE}  (must be False for JSON output)")

## 5. Load Qwen3 model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"Loading {MODEL_NAME} ...")
print(f"Quantisation: {'4-bit (BitsAndBytes)' if USE_4BIT else 'fp16'}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if USE_4BIT:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_cfg,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

model.eval()
print("✅ Model loaded")
mem = torch.cuda.memory_allocated() / 1024**3
print(f"   GPU memory used: {mem:.1f} GB")

## 6. Qwen3 LLM Client (disk cache + structured output)

In [ ]:
import hashlib, json as _json, re, time, logging, math
from typing import Dict, List, Optional
from pathlib import Path

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s")
logger = logging.getLogger(__name__)

# ── Disk Cache ────────────────────────────────────────────────
def _ckey(system, user):
    return hashlib.md5(f"{MODEL_NAME}||{system}||{user}".encode()).hexdigest()

def _cget(key):
    p = CACHE_DIR / f"{key}.json"
    try: return _json.loads(p.read_text("utf-8"))["r"] if p.exists() else None
    except: return None

def _cset(key, resp):
    (CACHE_DIR / f"{key}.json").write_text(
        _json.dumps({"r": resp}, ensure_ascii=False), "utf-8")


# ── Qwen3 Client (runs on local GPU) ─────────────────────────
class Qwen3Client:
    """
    Calls the locally loaded Qwen3 model.
    - Thinking mode OFF  → clean JSON output (required for parsing)
    - Disk cache to avoid re-running identical prompts across batches
    """
    model = MODEL_NAME.split("/")[-1]   # e.g. "Qwen3-8B"

    def call(self, system: str, user: str) -> str:
        key = _ckey(system, user)
        if USE_CACHE:
            cached = _cget(key)
            if cached: return cached

        messages = [
            {"role": "system",  "content": system},
            {"role": "user",    "content": user},
        ]

        # Apply chat template
        # thinking=False disables <think> blocks → clean JSON
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=THINKING_MODE,   # ← False = direct JSON output
        )

        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=1024,
                temperature=0.0,          # greedy (paper uses deterministic output)
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Decode only the new tokens (skip input prompt)
        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        response   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        if USE_CACHE: _cset(key, response)
        return response


# ── Response Parser ───────────────────────────────────────────
def parse_llm_response(response: str, expected_qids: List[str]) -> Dict[str, dict]:
    """
    Parse paper expected JSON: {"ranking": [{"qid": "Q123", "score": 5, "reason": "..."}]}

    Handles:
      - Clean JSON output (thinking=False)
      - Residual <think>...</think> blocks (if any)
      - Markdown code fences
      - Partial / malformed JSON → 4-stage fallback
    """
    valid = set(expected_qids)
    text  = response.strip()

    # Strip thinking blocks (safety — even with thinking=False Qwen3 can emit them)
    text = re.sub(r"<think>[\s\S]*?</think>", "", text).strip()
    # Strip code fences
    text = re.sub(r"```(?:json)?\s*", "", text).strip().rstrip("`").strip()

    def extract(obj):
        result, items = {}, []
        if isinstance(obj, dict):
            if "ranking" in obj:      items = obj["ranking"]
            elif "qid" in obj:        items = [obj]
            else:
                for k, v in obj.items():
                    if isinstance(v, dict) and "score" in v:
                        v["qid"] = k; items.append(v)
        elif isinstance(obj, list):
            items = obj
        for item in items:
            if not isinstance(item, dict): continue
            qid = str(item.get("qid", "")).strip().strip('\"\' ')
            if not qid.startswith("Q") or qid not in valid: continue
            try:   score = max(1.0, min(5.0, float(item.get("score", 3))))
            except: score = 3.0
            result[qid] = {"score": score, "reason": str(item.get("reason",""))[:300]}
        return result

    # Strategy 1: full JSON parse
    try:
        r = extract(_json.loads(text))
        if r: return r
    except: pass

    # Strategy 2: find JSON block
    for pat in [r'\{[^{}]*"ranking"[^{}]*\[[^\]]*\][^{}]*\}',
                r'\[[^\[\]]*"qid"[^\[\]]*\]',
                r'\{[^{}]*"qid"[^{}]*\}']:
        m = re.search(pat, text, re.DOTALL)
        if m:
            try:
                r = extract(_json.loads(m.group()))
                if r: return r
            except: pass

    # Strategy 3: individual objects
    result = {}
    for s in re.findall(r'\{[^{}]+\}', text):
        try: result.update(extract(_json.loads(s)))
        except: pass
    if result: return result

    # Strategy 4: neutral fallback
    logger.warning(f"Parse failed. Preview: {text[:150]}")
    return {q: {"score": 3.0, "reason": "parse_fallback"} for q in expected_qids}


# Instantiate
client = Qwen3Client()
print(f"✅ Qwen3Client ready  (model: {client.model}, cache: {USE_CACHE})")

# Smoke test
print("\nSmoke test (1 call) ...")
_test_resp = client.call(
    "You are a helpful assistant.",
    'Return exactly this JSON and nothing else: {"status": "ok"}'
)
print("Response:", _test_resp[:100])

## 7. Data loading

In [ ]:
def _safe(labels, idx):
    if not labels or idx >= len(labels): return "N/A"
    v = labels[idx]
    if v is None or (isinstance(v, float) and math.isnan(v)): return "N/A"
    return str(v).strip()

class HlinkData:
    def __init__(self, path=DATA_FILE):
        with open(path, encoding="utf-8") as f:
            self._raw = _json.load(f)
        self.languages    = list(self._raw.keys())
        self.article_qids = list(self._raw[self.languages[0]].keys())
        print(f"✅ {len(self.article_qids)} articles × {len(self.languages)} languages")

    def get_candidate_cards(self, article_qid, target_lang, top_n=TOP_N):
        lang_idx = LABEL_INDEX.get(target_lang, 0)
        art      = self._raw[target_lang].get(article_qid, {})
        if not art: return []
        sorted_hls = sorted(art["hyperlinks"].items(),
                            key=lambda x: x[1]["score"], reverse=True)[:top_n]
        cards = []
        for rank, (hqid, hinfo) in enumerate(sorted_hls, 1):
            lbs = hinfo.get("label", [])
            lang_scores = {}
            for lang in self.languages:
                e = self._raw[lang].get(article_qid,{}).get("hyperlinks",{}).get(hqid)
                if e: lang_scores[lang] = e["score"]
            lang_str = "+".join(l.upper() for l in self.languages if l in lang_scores)
            cards.append({
                "qid": hqid, "ngcf_rank": rank, "ngcf_score": hinfo["score"],
                "label_en": _safe(lbs,0), "label_vi": _safe(lbs,1), "label_ja": _safe(lbs,2),
                "label_tgt": _safe(lbs, lang_idx),
                "scores": lang_scores, "n_langs": len(lang_scores), "lang_str": lang_str,
            })
        return cards

    def get_article_info(self, article_qid, target_lang):
        lang_name = LANG_NAMES.get(target_lang, target_lang)
        labels    = {l: self._raw[l].get(article_qid,{}).get("label","N/A")
                     for l in self.languages}
        title     = labels.get(target_lang, article_qid)
        cards     = self.get_candidate_cards(article_qid, target_lang, top_n=20)
        agreed    = sorted(cards, key=lambda c:(c["n_langs"],c["ngcf_score"]), reverse=True)[:5]
        ctx_en    = ", ".join(c["label_en"]  for c in agreed if c["label_en"]  != "N/A")
        ctx_tgt   = ", ".join(c["label_tgt"] for c in agreed if c["label_tgt"] != "N/A")
        if target_lang == "en":
            lead = (f"{title} is a subject in English Wikipedia. "
                    f"Closely associated with: {ctx_en}.")
        else:
            lead = (f"{title} ({labels.get('en','N/A')} in English) "
                    f"is an article in {lang_name} Wikipedia. "
                    f"Related topics ({lang_name}): {ctx_tgt}. "
                    f"Related topics (English): {ctx_en}.")
        return title, lead

    def build_ground_truth(self, target_lang, held_rank=HELD_RANK, top_n=TOP_N):
        gt = {}
        for qid in self.article_qids:
            cards = self.get_candidate_cards(qid, target_lang, top_n)
            if len(cards) >= held_rank:
                gt[qid] = cards[held_rank-1]["qid"]
        return gt

    def article_label(self, qid, lang):
        return self._raw[lang].get(qid, {}).get("label", qid)

hdata = HlinkData()

## 8. Prompt builder (paper §3.5)

In [ ]:
def build_user_prompt(title, lead, candidates, target_lang, batch_num, total_batches):
    """
    User prompt matching paper §3.5:
      Input: Article title | Lead paragraph | Candidate hyperlinks
             Candidate descriptions | NGCF scores
      Return JSON only.
      Expected output: {"ranking": [{"qid":"Q...","score":5,"reason":"..."}]}
    """
    n = len(candidates)
    lines = []
    for c in candidates:
        lbl_en, lbl_tgt, score = c["label_en"], c["label_tgt"], c["ngcf_score"]
        n_l = c["n_langs"]
        desc = (f"Recommended by all 3 language NGCF models ({c['lang_str']})" if n_l==3 else
                f"Recommended by 2 language NGCF models ({c['lang_str']})"     if n_l==2 else
                f"Recommended by {c['lang_str']} NGCF model only")
        display = lbl_en if (target_lang=="en" or lbl_tgt==lbl_en or lbl_tgt=="N/A") \
                  else f"{lbl_tgt} ({lbl_en})"
        lines.append(f"  - QID: {c['qid']} | Label: {display} | "
                     f"NGCF score: {score:.2f} | Description: {desc}")

    return f"""Article:
{title}

Lead paragraph:
{lead}

Candidate list (batch {batch_num}/{total_batches}, {n} candidates):
{chr(10).join(lines)}

Return JSON only.
Expected output format:
{{"ranking": [{{"qid": "Q...", "score": 5, "reason": "Closely related to the article."}}, ...]}}

Score each candidate 1-5:
  5 = highly relevant   4 = relevant   3 = moderate   2 = weak   1 = irrelevant

Evaluate all {n} candidates. Return exactly {n} entries in the ranking array."""


# Preview prompt
title_p, lead_p = hdata.get_article_info("Q12557", "vi")
cards_p = hdata.get_candidate_cards("Q12557", "vi", top_n=TOP_N)
print("SYSTEM PROMPT:")
print(SYSTEM_PROMPT)
print()
print("USER PROMPT (first 3 candidates shown):")
print(build_user_prompt(title_p, lead_p, cards_p[:3], "vi", 1, 5))

## 9. Algorithm 1 — LLM-Assisted Reranking

In [ ]:
from tqdm.auto import tqdm

def minmax_norm(scores):
    if not scores: return {}
    lo, hi = min(scores.values()), max(scores.values())
    if hi == lo: return {k: 1.0 for k in scores}
    return {k: (v-lo)/(hi-lo) for k,v in scores.items()}

def fuse_scores(ngcf_norm, llm_norm, alpha=ALPHA):
    return {q: alpha*ngcf_norm.get(q,0.0) + (1-alpha)*llm_norm.get(q,0.0)
            for q in ngcf_norm}


class Algorithm1:
    """
    Paper Algorithm 1 — steps 1-10:
    Retrieve → Cards → Batch → LLM score → Aggregate →
    Norm NGCF → Norm LLM → Fuse → Sort → Top-K
    """
    def __init__(self, hdata, client, target_lang="vi",
                 top_n=TOP_N, batch_size=BATCH_SIZE,
                 alpha=ALPHA, final_k=FINAL_K):
        self.hdata, self.client = hdata, client
        self.target_lang = target_lang
        self.top_n, self.batch_size = top_n, batch_size
        self.alpha, self.final_k = alpha, final_k
        self.n_articles = self.n_llm_calls = self.n_parse_fails = 0

    def run_article(self, article_qid):
        self.n_articles += 1
        cards = self.hdata.get_candidate_cards(article_qid, self.target_lang, self.top_n)
        if not cards: return []
        title, lead = self.hdata.get_article_info(article_qid, self.target_lang)
        batches     = [cards[i:i+self.batch_size] for i in range(0,len(cards),self.batch_size)]
        n_b         = len(batches)
        all_llm     = {}

        for bi, batch in enumerate(batches):
            user_p   = build_user_prompt(title, lead, batch, self.target_lang, bi+1, n_b)
            response = self.client.call(SYSTEM_PROMPT, user_p)
            self.n_llm_calls += 1
            parsed   = parse_llm_response(response, [c["qid"] for c in batch])
            self.n_parse_fails += len([q for q in [c["qid"] for c in batch] if q not in parsed])
            all_llm.update(parsed)

        ngcf_raw  = {c["qid"]: c["ngcf_score"] for c in cards}
        llm_raw   = {q: v["score"] for q,v in all_llm.items()}
        ngcf_norm = minmax_norm(ngcf_raw)
        llm_norm  = minmax_norm(llm_raw) if llm_raw else {}
        fused     = fuse_scores(ngcf_norm, llm_norm, self.alpha)
        sorted_q  = sorted(fused, key=fused.get, reverse=True)
        card_map  = {c["qid"]: c for c in cards}

        results = []
        for qid in sorted_q[:self.final_k]:
            c   = card_map[qid]
            llm = all_llm.get(qid, {"score": None, "reason": ""})
            results.append({
                "qid": qid, "label_en": c["label_en"], "label_tgt": c["label_tgt"],
                "ngcf_rank": c["ngcf_rank"], "ngcf_score": round(c["ngcf_score"],3),
                "ngcf_norm": round(ngcf_norm.get(qid,0),6),
                "llm_score": llm["score"],
                "llm_norm":  round(llm_norm.get(qid,0),6),
                "fused_score": round(fused[qid],6),
                "reason": llm["reason"], "n_langs": c["n_langs"],
            })
        return results

    def run_all(self, article_qids):
        results = {}
        for qid in tqdm(article_qids, desc=f"Algo1 [{self.target_lang}]"):
            try:
                ranked = self.run_article(qid)
                results[qid] = [r["qid"] for r in ranked]
            except Exception as e:
                print(f"  ⚠️ {qid}: {e}")
                cards = self.hdata.get_candidate_cards(qid, self.target_lang, self.final_k)
                results[qid] = [c["qid"] for c in cards[:self.final_k]]
        return results

    def ngcf_baseline(self, article_qids):
        return {q: [c["qid"] for c in
                    self.hdata.get_candidate_cards(q, self.target_lang, self.final_k)]
                for q in article_qids}

    def stats(self):
        avg = self.n_llm_calls / max(self.n_articles, 1)
        print(f"Articles: {self.n_articles}  |  LLM calls: {self.n_llm_calls}"
              f"  ({avg:.1f}/article)  |  Parse fails: {self.n_parse_fails}")

print("✅ Algorithm1 defined")

## 10. Evaluation metrics

In [ ]:
import math

def hit_rate(ranked, gt, k): return 1.0 if gt in ranked[:k] else 0.0
def recall_k(ranked, gt, k): return hit_rate(ranked, gt, k)
def ndcg_k(ranked, gt, k):
    for pos, item in enumerate(ranked[:k], 1):
        if item == gt: return 1.0 / math.log2(pos+1)
    return 0.0

def compute_metrics(predictions, gt_map, k_values=K_VALUES):
    sums = {m: {k: 0.0 for k in k_values} for m in ["HR","Recall","NDCG"]}
    n = 0
    for art_qid, ranked in predictions.items():
        gt = gt_map.get(art_qid)
        if not gt: continue
        n += 1
        for k in k_values:
            sums["HR"][k]     += hit_rate(ranked, gt, k)
            sums["Recall"][k] += recall_k(ranked, gt, k)
            sums["NDCG"][k]   += ndcg_k(ranked, gt, k)
    if not n: return {"HR":{}, "Recall":{}, "NDCG":{}, "n":0}
    return {m: {k: round(sums[m][k]/n, 4) for k in k_values}
            for m in ["HR","Recall","NDCG"]} | {"n": n}

def print_report(report):
    s = report["settings"]
    print(f"\n{'='*68}")
    print(f"  Algorithm 1 — LLM-Assisted Hyperlink Reranking")
    print(f"  Language : {s['target_lang'].upper()}   Model : {s['llm_model']}")
    print(f"  NGCF top-{s['top_n']} → batch {s['batch_size']} → top-{s['final_k']}")
    print(f"  GT = NGCF rank-{s['held_rank']}  |  Articles : {s['n_articles']}")
    print(f"  LLM calls : {s['llm_calls']}  |  Best α : {report['best_alpha']}")
    print(f"{'='*68}")
    base     = report["ngcf_baseline"]
    systems  = [("NGCF baseline","ngcf_baseline")] +                [(f"Algo1 α={a}", f"algo1_alpha_{a}") for a in s["alpha_grid"]]
    for metric in ["HR","Recall","NDCG"]:
        print(f"\n  {metric}")
        print(f"  {'System':<22}" + "".join(f"  @{k:<5}" for k in s["k_values"]))
        print("  " + "─"*(22+8*len(s["k_values"])))
        for name, key in systems:
            vals = report.get(key,{}).get(metric,{})
            bv   = base.get(metric,{})
            row  = f"  {name:<22}"
            for k in s["k_values"]:
                v    = vals.get(k, 0.0)
                mark = "" if key=="ngcf_baseline" else ("↑" if v>bv.get(k,0) else ("↓" if v<bv.get(k,0) else "="))
                row += f"  {v:.4f}{mark}"
            print(row)
    print(f"\n  ↑/↓ vs NGCF baseline  |  n={report['ngcf_baseline']['n']} articles")
    print(f"{'='*68}\n")

print("✅ Metrics defined")

## 11. Demo — single article (Qwen3)

In [ ]:
DEMO_QID  = "Q12557"    # Đế quốc Mông Cổ / Mongol Empire
DEMO_LANG = "vi"

algo_demo = Algorithm1(hdata, client, target_lang=DEMO_LANG,
                       top_n=TOP_N, batch_size=BATCH_SIZE,
                       alpha=ALPHA, final_k=FINAL_K)

label = hdata.article_label(DEMO_QID, DEMO_LANG)
print(f"Running Algorithm 1 on {DEMO_QID} ({label}) [{DEMO_LANG}]...")
print(f"Model: {client.model}  |  top_n={TOP_N}  batch_size={BATCH_SIZE}  α={ALPHA}\n")

import time
t0      = time.time()
results = algo_demo.run_article(DEMO_QID)
elapsed = time.time() - t0
algo_demo.stats()
print(f"Time: {elapsed:.1f}s")

print(f"\nTop-{FINAL_K} fused recommendations:")
print(f"  {'#':<3}  {'QID':<12}  {'Label':<30}  {'NGCFr':>5}  {'NGCFn':>6}  {'LLM':>4}  {'LLMn':>6}  {'Fused':>7}")
print("  " + "─"*82)
for i, r in enumerate(results, 1):
    llm_s = f"{r['llm_score']:.0f}" if r["llm_score"] is not None else "—"
    print(f"  {i:<3}  {r['qid']:<12}  {r['label_en']:<30}  "
          f"{r['ngcf_rank']:>5}  {r['ngcf_norm']:>6.3f}  {llm_s:>4}  "
          f"{r['llm_norm']:>6.3f}  {r['fused_score']:>7.4f}")
    if r["reason"]:
        print(f"       └─ {r['reason'][:80]}")

## 12. Full evaluation — all 280 articles, α ablation

> **Estimated time:**  
> - Qwen3-8B 4-bit on T4: ~3–5 min/article × 280 = **14–24 hours** for full run  
> - Use `MAX_ARTICLES = 30` for a quick representative evaluation (~1.5–2 hours)  
> - Cache is saved to disk — safe to interrupt and resume  

In [ ]:
# ── Settings ─────────────────────────────────────────────────
EVAL_LANG     = "vi"    # "en" / "vi" / "ja"
MAX_ARTICLES  = 30      # ← None = all 280; 30 for quick test (~1.5 h on T4)
EVAL_ALPHA_GRID = ALPHA_GRID

# ── Ground truth ─────────────────────────────────────────────
gt_map   = hdata.build_ground_truth(EVAL_LANG, HELD_RANK, TOP_N)
art_qids = list(gt_map.keys())
if MAX_ARTICLES:
    art_qids = art_qids[:MAX_ARTICLES]
    gt_map   = {q: gt_map[q] for q in art_qids}
print(f"Evaluating {len(art_qids)} articles | lang={EVAL_LANG} | GT=rank-{HELD_RANK}")

# ── NGCF baseline ────────────────────────────────────────────
eval_algo  = Algorithm1(hdata, client, EVAL_LANG, TOP_N, BATCH_SIZE, ALPHA, FINAL_K)
ngcf_preds = eval_algo.ngcf_baseline(art_qids)
ngcf_met   = compute_metrics(ngcf_preds, gt_map, K_VALUES)
print(f"NGCF baseline done. HR@10 = {ngcf_met['HR'].get(10, 0):.4f}")

# ── Algorithm 1: collect raw scores once (LLM called here) ──
raw_store = {}
import time as _time

print(f"\nRunning LLM scoring ({len(art_qids)} articles × "
      f"~{math.ceil(TOP_N/BATCH_SIZE)} batches = "
      f"~{len(art_qids)*math.ceil(TOP_N/BATCH_SIZE)} LLM calls) ...")
print("Cache is active — interrupted runs can be safely resumed.\n")

t_start = _time.time()
for idx, art_qid in enumerate(tqdm(art_qids, desc=f"Qwen3 [{EVAL_LANG}]")):
    cards       = hdata.get_candidate_cards(art_qid, EVAL_LANG, TOP_N)
    title, lead = hdata.get_article_info(art_qid, EVAL_LANG)
    batches_    = [cards[j:j+BATCH_SIZE] for j in range(0, len(cards), BATCH_SIZE)]
    all_llm_    = {}
    for bi, batch in enumerate(batches_):
        user_p = build_user_prompt(title, lead, batch, EVAL_LANG, bi+1, len(batches_))
        resp   = client.call(SYSTEM_PROMPT, user_p)
        eval_algo.n_llm_calls += 1
        all_llm_.update(parse_llm_response(resp, [c["qid"] for c in batch]))
    raw_store[art_qid] = {
        c["qid"]: {"ngcf": c["ngcf_score"],
                   "llm":  all_llm_.get(c["qid"], {"score": 3.0})["score"]}
        for c in cards
    }
    # ETA
    if idx > 0 and idx % 5 == 0:
        elapsed_ = _time.time() - t_start
        eta_     = elapsed_ / (idx+1) * (len(art_qids) - idx - 1)
        print(f"  [{idx+1}/{len(art_qids)}] elapsed {elapsed_/60:.1f} min | "
              f"ETA {eta_/60:.1f} min | LLM calls: {eval_algo.n_llm_calls}")

# ── α ablation ───────────────────────────────────────────────
alpha_metrics = {}
for alpha in EVAL_ALPHA_GRID:
    preds = {}
    for art_qid, scores in raw_store.items():
        ng = {q: v["ngcf"] for q,v in scores.items()}
        ll = {q: v["llm"]  for q,v in scores.items()}
        fs = fuse_scores(minmax_norm(ng), minmax_norm(ll), alpha)
        preds[art_qid] = sorted(fs, key=fs.get, reverse=True)[:FINAL_K]
    alpha_metrics[alpha] = compute_metrics(preds, gt_map, K_VALUES)

best_alpha = max(EVAL_ALPHA_GRID,
                 key=lambda a: alpha_metrics[a].get("NDCG",{}).get(FINAL_K,0))

# ── Build report ─────────────────────────────────────────────
report = {
    "settings": {
        "target_lang": EVAL_LANG, "held_rank": HELD_RANK, "top_n": TOP_N,
        "batch_size": BATCH_SIZE, "final_k": FINAL_K, "k_values": K_VALUES,
        "alpha_grid": EVAL_ALPHA_GRID, "n_articles": len(art_qids),
        "llm_model":  client.model,
        "llm_calls":  eval_algo.n_llm_calls,
        "parse_fails":eval_algo.n_parse_fails,
    },
    "ngcf_baseline": ngcf_met,
    "best_alpha":    best_alpha,
}
for a in EVAL_ALPHA_GRID:
    report[f"algo1_alpha_{a}"] = alpha_metrics[a]
report["algo1_best"] = alpha_metrics[best_alpha]

print_report(report)

## 13. Save results & plot

In [ ]:
import json as _js

# Save JSON
model_slug = client.model.replace("/","-").replace(":","-")
fname = OUTPUT_DIR / f"algo1_{EVAL_LANG}_rank{HELD_RANK}_top{TOP_N}_b{BATCH_SIZE}_{model_slug}.json"
fname.write_text(_js.dumps(report, indent=2, ensure_ascii=False), "utf-8")
print(f"✅ Report saved → {fname}")

# Download from Colab
from google.colab import files as colab_files
colab_files.download(str(fname))

# ── Plot ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams.update({"figure.dpi":110, "axes.spines.top":False,
                              "axes.spines.right":False})
k_vals   = K_VALUES
systems  = {"NGCF": "ngcf_baseline",
             **{f"Algo1 α={a}": f"algo1_alpha_{a}" for a in EVAL_ALPHA_GRID}}
colors   = ["#888", "#2a78d6", "#1baf7a", "#e05c1a"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
fig.suptitle(
    f"Algorithm 1 — {EVAL_LANG.upper()} | {client.model} | {len(art_qids)} articles",
    fontsize=13, y=1.02
)
for ax, metric in zip(axes, ["HR","Recall","NDCG"]):
    x   = range(len(k_vals))
    w   = 0.18
    off = -(len(systems)-1)/2 * w
    for (name, key), color in zip(systems.items(), colors):
        vals = [report.get(key,{}).get(metric,{}).get(k,0) for k in k_vals]
        ax.bar([xi+off for xi in x], vals, width=w, label=name, color=color, alpha=0.85)
        off += w
    ax.set_xticks(list(x))
    ax.set_xticklabels([f"@{k}" for k in k_vals])
    ax.set_title(metric)
    ax.set_ylim(0, max(0.05, ax.get_ylim()[1]*1.15))
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(matplotlib.ticker.FormatStrFormatter("%.3f"))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"results_{EVAL_LANG}.png", bbox_inches="tight")
plt.show()

## 14. (Optional) All 3 languages

In [ ]:
# Run all 3 languages sequentially.
# Each takes ~30 min with MAX_ARTICLES=30 on T4.

# MULTI_MAX = 30    # articles per language
# all_reports = {}
#
# for lang in ["en", "vi", "ja"]:
#     print(f"\n{'='*55}\n  Language: {lang}\n{'='*55}")
#     gt    = hdata.build_ground_truth(lang, HELD_RANK, TOP_N)
#     qids  = list(gt.keys())[:MULTI_MAX]
#     gt    = {q: gt[q] for q in qids}
#
#     a_    = Algorithm1(hdata, client, lang, TOP_N, BATCH_SIZE, ALPHA, FINAL_K)
#     ng_p  = a_.ngcf_baseline(qids)
#     ng_m  = compute_metrics(ng_p, gt, K_VALUES)
#
#     raw_  = {}
#     for aqid in tqdm(qids, desc=lang):
#         cards_ = hdata.get_candidate_cards(aqid, lang, TOP_N)
#         t_,l_  = hdata.get_article_info(aqid, lang)
#         b_     = [cards_[j:j+BATCH_SIZE] for j in range(0,len(cards_),BATCH_SIZE)]
#         ll_    = {}
#         for bi,bch in enumerate(b_):
#             up = build_user_prompt(t_,l_,bch,lang,bi+1,len(b_))
#             ll_.update(parse_llm_response(client.call(SYSTEM_PROMPT,up),
#                                           [c["qid"] for c in bch]))
#         raw_[aqid] = {c["qid"]:{"ngcf":c["ngcf_score"],
#                                  "llm":ll_.get(c["qid"],{"score":3.0})["score"]}
#                       for c in cards_}
#
#     am_ = {}
#     for alpha in ALPHA_GRID:
#         preds_ = {}
#         for aqid,sc in raw_.items():
#             fs_ = fuse_scores(minmax_norm({q:v["ngcf"] for q,v in sc.items()}),
#                               minmax_norm({q:v["llm"]  for q,v in sc.items()}), alpha)
#             preds_[aqid] = sorted(fs_,key=fs_.get,reverse=True)[:FINAL_K]
#         am_[alpha] = compute_metrics(preds_,gt,K_VALUES)
#     ba_ = max(ALPHA_GRID, key=lambda a:am_[a].get("NDCG",{}).get(FINAL_K,0))
#
#     rep_ = {"settings":{"target_lang":lang,"held_rank":HELD_RANK,"top_n":TOP_N,
#                          "batch_size":BATCH_SIZE,"final_k":FINAL_K,"k_values":K_VALUES,
#                          "alpha_grid":ALPHA_GRID,"n_articles":len(qids),
#                          "llm_model":client.model,"llm_calls":a_.n_llm_calls,
#                          "parse_fails":a_.n_parse_fails},
#              "ngcf_baseline":ng_m,"best_alpha":ba_}
#     for a in ALPHA_GRID: rep_[f"algo1_alpha_{a}"] = am_[a]
#     rep_["algo1_best"] = am_[ba_]
#     all_reports[lang] = rep_
#     print_report(rep_)
#
#     out_ = OUTPUT_DIR/f"algo1_{lang}_rank{HELD_RANK}_top{TOP_N}_b{BATCH_SIZE}_{client.model.replace('/','_')}.json"
#     out_.write_text(_js.dumps(rep_,indent=2,ensure_ascii=False),"utf-8")
#     colab_files.download(str(out_))

print("Uncomment block above to run all 3 languages.")